# Quantization Inference Server (Colab T4 GPU + Ngrok)


In [ ]:
!nvidia-smi
!pip install -q fastapi uvicorn pydantic pyngrok nvidia-ml-py psutil huggingface_hub
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir


In [ ]:
import os
from huggingface_hub import hf_hub_download

HF_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
MODELS_DIR = "./models"
os.makedirs(MODELS_DIR, exist_ok=True)

models = [
    "qwen2.5-1.5b-instruct-q4_k_m.gguf",
    "qwen2.5-1.5b-instruct-q8_0.gguf",
    "qwen2.5-1.5b-instruct-fp16.gguf"
]

for fname in models:
    target = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(target):
        hf_hub_download(repo_id=HF_REPO, filename=fname, local_dir=MODELS_DIR)


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import subprocess, time

log_f = open("server.log", "w")
proc = subprocess.Popen(["uvicorn", "quant_server:app", "--host", "0.0.0.0", "--port", "8000"],
                        stdout=log_f, stderr=subprocess.STDOUT)
time.sleep(6)
print(open("server.log").read())


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3Irxz7tUedAHT6UftwwVa2BVNeR_3TJ5ZhmbB4GosNNaLyVjs")
ngrok.kill()
public_url = ngrok.connect(8000, "http")
print("Public URL:", public_url)
